In [1]:
# Import file
import numpy as np
import pandas as pd
import requests
import time
import re
import pickle
import os

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

In [163]:
# Load Data
df = pd.read_csv("books.csv", encoding="cp1252")

## Preprocessing

In [164]:
df.head(2)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,NaN
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,09-01-2004,Scholastic Inc.,NaN


In [165]:
df.columns

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher', 'Unnamed: 12'],
      dtype='object')

In [ ]:
import asyncio
import aiohttp
import pandas as pd
import os

df = pd.read_csv("books.csv", encoding="latin-1")
df["isbn13"] = df["isbn13"].astype(str).str.replace(".0", "", regex=False).str.strip()

CHECKPOINT_FILE = "enrichment_checkpoint.csv"
CONCURRENCY = 20  # 20 requests at a time

# Resume from checkpoint
if os.path.exists(CHECKPOINT_FILE):
    done_df = pd.read_csv(CHECKPOINT_FILE)
    completed_isbns = set(done_df["isbn13"].astype(str))
    results = done_df.to_dict("records")
    print(f"Resuming — {len(completed_isbns)} already done")
else:
    completed_isbns = set()
    results = []

async def fetch_book(session, row, semaphore):
    isbn = str(row["isbn13"])
    url = f"https://openlibrary.org/api/books?bibkeys=ISBN:{isbn}&format=json&jscmd=data"

    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=10)) as resp:
                data = await resp.json(content_type=None)
                key = f"ISBN:{isbn}"

                if key in data:
                    book_data = data[key]
                    subjects = book_data.get("subjects", [])
                    genre = ", ".join([s["name"] for s in subjects[:5]])
                    description = book_data.get("notes", "") or \
                                  book_data.get("subtitle", "No Description")
                else:
                    genre, description = "Unknown", "No Description"

        except Exception as e:
            genre, description = "Error", "Error"

    return {**row.to_dict(), "genre": genre, "description": description}

async def main():
    semaphore = asyncio.Semaphore(CONCURRENCY)
    rows_to_fetch = [row for _, row in df.iterrows() 
                     if str(row["isbn13"]) not in completed_isbns]
    
    print(f"Fetching {len(rows_to_fetch)} books with {CONCURRENCY} concurrent requests...")

    async with aiohttp.ClientSession() as session:
        tasks = [fetch_book(session, row, semaphore) for row in rows_to_fetch]

        for i, coro in enumerate(asyncio.as_completed(tasks)):
            result = await coro
            results.append(result)

            if (i + 1) % 200 == 0:
                pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
                print(f"  {i + 1}/{len(rows_to_fetch)} done...")

    pd.DataFrame(results).to_csv("books_enriched.csv", index=False)
    print(f"Done — {len(results)} rows saved")

# Run it
await main()

In [3]:
data = pd.read_csv("books_enriched.csv")

In [4]:
data.head(1)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12,genre,description
0,12249,The New Testament,Richmond Lattimore,4.32,0865475245,9780865475243,eng,608,109,23,10/30/1997,North Point Press,NaN,"Bible, criticism, interpretation, etc., n. t.",No Description


In [5]:
data.head(1)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12,genre,description
0,12249,The New Testament,Richmond Lattimore,4.32,0865475245,9780865475243,eng,608,109,23,10/30/1997,North Point Press,NaN,"Bible, criticism, interpretation, etc., n. t.",No Description


In [6]:
data["description"].nunique()

4490

In [7]:
data["description"].value_counts()

description
No Description                                          5434
Includes bibliographical references and index.           174
Includes index.                                          158
a novel                                                  136
Includes bibliographical references.                     106
                                                        ... 
Never Despair, 1945-1965                                   1
Includes bibliographical references (p. 510).              1
"A welcome book."                                          1
notes.                                                     1
My Encounter With Marx And Freud (Continuum Impacts)       1
Name: count, Length: 4490, dtype: int64

In [8]:
data["genre"].value_counts()

genre
Unknown                                                                                                                                                    687
Error                                                                                                                                                       41
Fiction, general                                                                                                                                            25
Fiction, fantasy, general                                                                                                                                   25
Fiction, science fiction, general                                                                                                                           17
                                                                                                                                                          ... 
Psychoanalysis, Listening, Psychological

In [9]:
def clean_genre(genre_str):
    """ Get the top 1 genre for the book or
        Blank for the null values"""
    if pd.isna(genre_str) or genre_str in ("Unknown", "Error", ""):
        return ""  # empty string, not "Unknown"
    
    first = str(genre_str).split(",")[0].strip().title()
    return first

data["genre_clean"] = data["genre"].apply(clean_genre)

In [10]:
def clean_genre(genre_str):
    """ Get the top 1 genre for the book or
        Blank for the null values"""
    if pd.isna(genre_str) or genre_str in ("Unknown", "Error", ""):
        return ""  # empty string, not "Unknown"
    
    first = str(genre_str).split(",")[0].strip().title()
    return first

data["genre_clean"] = data["genre"].apply(clean_genre)

In [11]:
data.head(5)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12,genre,description,genre_clean
0,12249,The New Testament,Richmond Lattimore,4.32,0865475245,9780865475243,eng,608,109,23,10/30/1997,North Point Press,NaN,"Bible, criticism, interpretation, etc., n. t.",No Description,Bible
1,30651,The Art of Loving by Erich Fromm: A True Story...,Lala Okamoto,1.67,4990327500,9784990327507,eng,227,3,2,10/31/2006,Intercultural Publishing,NaN,NaN,A True Story of a Japanese Woman,
2,30639,Mere Christianity: Abolition of Man (Bonus Fea...,C.S. Lewis/Geoffrey Howard/Robert Whitefield/R...,4.32,0786174366,9780786174362,eng,6,43,6,05-01-2006,Blackstone Audiobooks,NaN,"Apologetics, Doctrinal Theology, Popular works...",No Description,Apologetics
3,30633,The Four Loves,C.S. Lewis,4.14,0006280897,9780006280897,en-US,170,34681,1084,06-05-2002,HarperCollins Publishers Ltd,NaN,"Love (Theology), Love, Religious aspects of Lo...",No Description,Love (Theology)
4,12240,The Play Soldier,Chet Green,4.5,159113644X,9781591136446,eng,328,2,2,5/22/2011,Booklocker.com Inc.,NaN,NaN,No Description,


In [12]:
data.isnull().sum()

bookID                    0
title                     0
authors                   0
average_rating            0
isbn                      0
isbn13                    0
language_code             0
  num_pages               0
ratings_count             0
text_reviews_count        0
publication_date          0
publisher                 0
Unnamed: 12           11123
genre                   511
description               0
genre_clean               0
dtype: int64

In [13]:
data.columns

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher', 'Unnamed: 12', 'genre', 'description',
       'genre_clean'],
      dtype='object')

In [179]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11127 entries, 0 to 11126
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           11127 non-null  object
 1   authors         11127 non-null  object
 2   average_rating  11127 non-null  object
 3   isbn13          11127 non-null  object
 4   language_code   11127 non-null  object
 5     num_pages     11127 non-null  object
 6   ratings_count   11127 non-null  int64 
 7   description     11127 non-null  object
 8   genre_clean     11127 non-null  object
dtypes: int64(1), object(8)
memory usage: 782.5+ KB


## Feature engineering

In [180]:
# create combined features
data["combined_features"] = (
    data["title"].fillna('') + " " +
    data["authors"].fillna('') + " " +
    data["genre_clean"].fillna('') + " " +
    data["description"].fillna('')
)

In [181]:
data["average_rating"] = pd.to_numeric(
    data["average_rating"],
    errors="coerce"
)

In [182]:
# Lowercaseing       # hyperparameter
data["combined_features"] = data["combined_features"].str.lower()

In [183]:
# Remove puncutation          # Hyperparameter
def clean_text(text):

    text = re.sub(r'[^a-zA-Z0-9\s]', '', str(text))

    return text

data["combined_features"] = data["combined_features"].apply(clean_text)

In [184]:
# Convert text to num
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1,2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(data["combined_features"])

### Model KNN (Better)

In [185]:
# Implementation
knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

knn.fit(tfidf_matrix)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [189]:
def recommend_books(
    book_title=None,
    author=None,
    genre=None,
    min_rating=0.0,
    language="eng",
    n_recommendations=5
):
    """
    Recommend books based on:
    - book_title  : find books similar to this title
    - author      : find books by or similar to this author
    - genre       : filter by genre
    - min_rating  : minimum average rating (0.0 to 5.0)
    - language    : language code (default 'eng')
    - n_recommendations: how many to return
    """

    # ── Step 1: start with full dataset, apply filters ──
    filtered = data.copy()

    if language:
        filtered = filtered[filtered["language_code"].str.strip() == language]

    if min_rating > 0.0:
        filtered = filtered[filtered["average_rating"] >= min_rating]

    if genre:
        filtered = filtered[
            filtered["genre_clean"].str.contains(genre, case=False, na=False)
        ]

    if filtered.empty:
        return "No books found with those filters."

    # ── Step 2: content-based path (title or author given) ──
    if book_title or author:

        # find seed book(s)
        if book_title:
            seed = filtered[filtered["title"].str.contains(book_title, case=False, na=False)]
        else:
            seed = filtered[filtered["authors"].str.contains(author, case=False, na=False)]

        if seed.empty:
            return "No matching book found — try relaxing filters (e.g. lower min_rating)."

        # use first match as seed
        idx = seed.index[0]

        # knn on full tfidf_matrix, then filter results after
        distances, indices = knn.kneighbors(
            tfidf_matrix[idx],
            n_neighbors=len(data)  # fetch all, filter below
        )

        # filter neighbours to only those in our filtered subset
        filtered_indices = [
            i for i in indices.flatten()
            if i != idx and i in filtered.index
        ][:n_recommendations]

        if not filtered_indices:
            return "No recommendations found with those filters."

        recommendations = data.iloc[filtered_indices].copy()

    # ── Step 3: genre/filter-only path (no title or author) ──
    else:
        # sort by bayesian score — rewards popular AND highly rated books
        C = data["average_rating"].mean()   # global mean rating
        m = data["ratings_count"].quantile(0.75)  # minimum votes threshold

        def bayesian_score(row):
            v = row["ratings_count"]
            R = row["average_rating"]
            return (v / (v + m)) * R + (m / (v + m)) * C

        filtered = filtered.copy()
        filtered["score"] = filtered.apply(bayesian_score, axis=1)
        recommendations = filtered.sort_values("score", ascending=False).head(n_recommendations)

    # ── Step 4: display ──
    from IPython.display import display, HTML

    cards = ""
    for _, row in recommendations.iterrows():
        cover = f"https://covers.openlibrary.org/b/isbn/{row['isbn13']}-M.jpg"
        cards += f"""
        <div style="display:inline-block; margin:10px; text-align:center; 
                    width:130px; vertical-align:top">
            <img src="{cover}" width="100" height="150"
                 style="object-fit:cover; border-radius:4px; border:1px solid #ddd"
                 onerror="this.src='https://via.placeholder.com/100x150?text=No+Cover'"/>
            <p style="font-size:11px; margin:4px 0"><b>{row['title'][:40]}</b></p>
            <p style="font-size:11px; margin:0; color:gray">{row['authors'][:30]}</p>
            <p style="font-size:11px; margin:0">⭐ {row['average_rating']}</p>
            <p style="font-size:11px; margin:0; color:#888">{row.get('genre_clean', '')}</p>
        </div>
        """

    display(HTML(f"<div style='display:flex; flex-wrap:wrap'>{cards}</div>"))

In [ ]:
recommend_books(book_title="Dune", genre="Science Fiction", min_rating=3.0, n_recommendations=5)

In [ ]:
os.makedirs("models", exist_ok=True)

# 1. The KNN model
with open("models/model.pkl", "wb") as f:
    pickle.dump(knn, f)

# 2. Book titles — for search and frontend dropdowns
with open("models/book_names.pkl", "wb") as f:
    pickle.dump(data["title"].tolist(), f)

# 3. Ratings data — for displaying results
with open("models/final_rating.pkl", "wb") as f:
    pickle.dump(data[["title", "authors", "average_rating", 
                       "ratings_count", "isbn13", "genre_clean"]], f)

# 4. The TF-IDF matrix — for similarity lookup
with open("models/book_pivot.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

# 5. Vectorizer — don't forget this one
with open("modelsmodels/vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("All artifacts saved.")

All artifacts saved.


In [199]:
with open("artifacts/model.pkl", "rb") as f:
    knn = pickle.load(f)

with open("artifacts/book_names.pkl", "rb") as f:
    book_names = pickle.load(f)

with open("artifacts/final_rating.pkl", "rb") as f:
    final_rating = pickle.load(f)

with open("artifacts/book_pivot.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)

with open("artifacts/vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

In [200]:
recommend_books(book_title="Dune", genre="Science Fiction", min_rating=3.0, n_recommendations=5)